Este script implementa a técnica de Self-Consistency para gerar os escores contínuos necessários para a calibração multiplicativa.

In [1]:
import pandas as pd
import nltk
from openai import OpenAI
from tqdm import tqdm

nltk.download('punkt_tab', quiet=True)

True

In [2]:
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
MODELO_GUARD = "PatronusAI/lynx3_4b_full_finetune_v0.995-fp8-dynamic"

def auditar_com_votos(csv_relatorios, k=5):
    df = pd.read_csv(csv_relatorios)
    resultados_sentencas = []

    print(f"Iniciando auditoria de {len(df)} relatórios (K={k} votos por sentença)...")

    for _, row in tqdm(df.iterrows(), total=len(df)):
        contexto = row['contexto_completo']
        relatorio = row['relatorio_ia']
        sentencas = nltk.sent_tokenize(relatorio)

        for sent in sentencas:
            votos_unfaithful = 0
            prompt_lynx = f"Context: {contexto}\n\nStatement: {sent}\n\nIs the statement faithful to the context?"
            
            for _ in range(k):
                try:
                    res = client.chat.completions.create(
                        model=MODELO_GUARD,
                        messages=[{"role": "user", "content": prompt_lynx}],
                        temperature=0.7 # Variabilidade para o consenso
                    )
                    veredito = res.choices[0].message.content.lower()
                    if "unfaithful" in veredito or "hallucination" in veredito:
                        votos_unfaithful += 1
                except:
                    continue
            
            # s_i: escore contínuo entre 0.0 e 1.0
            si_score = votos_unfaithful / k
            
            resultados_sentencas.append({
                "id_bo": row['codigo_bo'],
                "sentenca": sent,
                "non_conformity_score": si_score
            })

    df_sent = pd.DataFrame(resultados_sentencas)
    df_sent.to_csv("sentencas_auditadas_votos.csv", index=False)
    print("\nAuditoria concluída.")

In [3]:
auditar_com_votos("dataset_com_relatorios.csv")

Iniciando auditoria de 200 relatórios (K=5 votos por sentença)...


  0%|          | 0/200 [00:00<?, ?it/s]

100%|██████████| 200/200 [4:08:10<00:00, 74.45s/it]   


Auditoria concluída.
